# WASP-52b: EXHALE model vs. observed He\,10830 and H$\alpha$ (L1-truncated domain)

Simple **overplot, no fitting**. EXHALE escape model: $\epsilon$\,Eri MUSCLES SED
scaled to $a=0.0272$ au, Yan et al. (2022) parameters, pressure (1 $\mu$bar) base BC,
full **Roche (tidal) potential truncated at the L1/Hill radius** ($r_{\max}\simeq2.53\,R_p$).
Transmission from `TPM.py`.

The earlier 10 $R_p$ extended-domain runs over-predicted He\,10830 by $\sim$3--10$\times$
because the outer He\,2$^3$S tail beyond $L_1$ (Roche-unbound, physically funneled
through $L_1$, not a spherical shell) dominates the impact-parameter ($r^2$-weighted)
absorption. Truncating at $L_1$ removes that artifact.

**Flux mapping** ($\times$0.5 from the 2-D method): our $0.25\times,0.5\times,1.0\times$
fiducial SED $=$ Yan's $0.5\,F_0,\,1\,F_0,\,2\,F_0$. Yan's He-fit is $0.5\,F_0$ + He-poor.
Grid: 3 fluxes $\times$ \{solar 92/8, He-poor 98/2, solar+metals\}.

Conventions: He is *excess absorption* (positive); H$\alpha$ is *transit spectrum*
(negative dip). He observation converted vacuum$\to$air. **H$\alpha$ is unreliable here
(Ly$\alpha$ RT not solved) and shown only for completeness.**

In [1]:
import os, numpy as np, matplotlib.pyplot as plt

HERE = os.path.abspath('.')
OBS  = '/home/kiseon/Exoplanetary_Atmosphere/WASP-52b'

def vac_to_air(lam_vac):
    s = 1.0e4 / lam_vac
    n = 1.0 + 0.0000834254 + 0.02406147/(130.0 - s**2) + 0.00015998/(38.9 - s**2)
    return lam_vac / n

he      = np.loadtxt(os.path.join(OBS, 'wasp52_He10830_trans_spec.dat'))   # lam_VAC, exc%, sig
he_lair = vac_to_air(he[:, 0])
ha      = np.loadtxt(os.path.join(OBS, 'WASP52b_Halpha_transpec_0p10_AA_binned.txt'))

# L1-truncated cases: (folder, label, color, linestyle).  color=composition, ls=flux(F0)
cases = [
    ('fxuv0p25_solar_L1',     r'solar, $0.5F_0$',      'C0', '-'),
    ('fxuv0p5_solar_L1',      r'solar, $1F_0$',        'C0', '--'),
    ('fxuv1p0_solar_L1',      r'solar, $2F_0$',        'C0', ':'),
    ('fxuv0p25_he98_L1',      r'He 98/2, $0.5F_0$',    'C1', '-'),
    ('fxuv0p5_he98_L1',       r'He 98/2, $1F_0$',      'C1', '--'),
    ('fxuv1p0_he98_L1',       r'He 98/2, $2F_0$',      'C1', ':'),
    ('fxuv0p5_solar_met_L1',  r'solar+metals, $1F_0$', 'C2', '--'),
    ('fxuv1p0_solar_met_L1',  r'solar+metals, $2F_0$', 'C2', ':'),
]

def load_model(folder, line):
    f = os.path.join(HERE, folder, 'tpm_%s.txt' % line)
    if not os.path.exists(f):
        return None
    d = np.loadtxt(f)
    return d[:, 0], (1.0 - d[:, 3]) * 100.0

In [2]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))

# ---- He I 10830 (transit depth, negative=absorption; air scale) ----
ax[0].errorbar(he_lair, -he[:,1], yerr=he[:,2], fmt='o', ms=3, color='0.4',
               ecolor='0.7', capsize=2, label='CARMENES (obs, vac$\\to$air)', zorder=1)
for folder, lab, col, ls in cases:
    m = load_model(folder, 'He10830')
    if m is None: print('missing He:', folder); continue
    ax[0].plot(m[0], -m[1], color=col, ls=ls, lw=1.8, label=lab, zorder=3)
ax[0].axhline(0, color='0.6', lw=0.8)
ax[0].set_xlim(10828.0, 10832.0)
ax[0].set_xlabel(r'Wavelength [\AA] (air, planet rest frame)')
ax[0].set_ylabel(r'Transit depth [\%]  (negative $=$ absorption)')
ax[0].set_title(r'He\,\textsc{i} 10830  ($L_1$-truncated)')
ax[0].legend(fontsize=7, ncol=2)

# ---- H-alpha (transit spectrum, negative dip; UNRELIABLE - no Lya RT) ----
ax[1].errorbar(ha[:,0], ha[:,1]*100.0, yerr=ha[:,2]*100.0, fmt='o', ms=3, color='0.4',
               ecolor='0.7', capsize=2, label='obs (binned)', zorder=1)
for folder, lab, col, ls in cases:
    m = load_model(folder, 'Halpha')
    if m is None: print('missing Ha:', folder); continue
    ax[1].plot(m[0], -m[1], color=col, ls=ls, lw=1.8, label=lab, zorder=3)
ax[1].axhline(0, color='0.6', lw=0.8)
ax[1].set_xlim(6561.0, 6564.6)
ax[1].set_xlabel(r'Wavelength [\AA] (air)')
ax[1].set_ylabel(r'Transit depth [\%]  (negative $=$ absorption)')
ax[1].set_title(r'H$\alpha$ 6562.8  (unreliable: no Ly$\alpha$ RT)')
ax[1].legend(fontsize=7, ncol=2)

fig.suptitle(r'WASP-52b: EXHALE (Roche tidal, $L_1$-truncated $\simeq2.53\,R_p$) vs.\ observation --- no fitting')
fig.tight_layout()
fig.savefig('compare_WASP-52b_L1.png', dpi=140)
plt.show()
print('saved compare_WASP-52b_L1.png')

saved compare_WASP-52b_L1.png


## Atmospheric structure (L1-truncated)

In [3]:
# T, v, n profiles (L1-truncated domain, r=1..2.53 Rp)
fig, ax = plt.subplots(1, 3, figsize=(15, 4.6))
for folder, lab, col, ls in cases:
    f = os.path.join(HERE, folder, 'output', 'Hydro_ioniz_adv.txt')
    if not os.path.exists(f): continue
    d = np.loadtxt(f); r, n, v, T = d[:,0], d[:,1], d[:,2], d[:,4]
    kw = dict(color=col, ls=ls, lw=1.6, label=lab)
    ax[0].plot(r, T, **kw); ax[1].plot(r, v/1e5, **kw); ax[2].semilogy(r, n, **kw)
ax[0].set_ylabel(r'$T$ [K]'); ax[0].set_title('Temperature')
ax[1].set_ylabel(r'$v$ [km s$^{-1}$]'); ax[1].set_title('Velocity'); ax[1].axhline(0, color='0.6', lw=0.8)
ax[2].set_ylabel(r'$n$ [cm$^{-3}$]'); ax[2].set_title('Total number density')
for a in ax:
    a.set_xlabel(r'$R/R_{\rm pl}$'); a.grid(alpha=0.3, which='both'); a.set_xlim(1, 2.53)
ax[0].legend(fontsize=7, ncol=2)
fig.suptitle(r'WASP-52b EXHALE structure (Roche tidal, $L_1$-truncated)')
fig.tight_layout()
plt.show()